# SaaS Gold Layer -- Analyst Exploration Notebook

**Purpose:** Explore the fully processed gold tables for our SaaS company dataset.  
**Data source:** `sv_ai_builder_workspace_catalog.saas_gold_dev`  
**Analysis workspace:** `sv_ai_builder_workspace_catalog.saas_analysis_sandbox`  
**Compute:** Databricks Connect (Serverless)

---

## Gold Table Inventory

| Table | Type | Description |
|-------|------|-------------|
| `dim_user` | Dimension (SCD2) | Current and historical user attributes -- subscription tier, product, location |
| `fact_order` | Fact (SCD1) | One row per order at latest state -- revenue, status, product mix |
| `fact_event` | Fact (SCD1) | One row per deduplicated event -- usage, errors, feature adoption |
| `agg_daily_revenue` | Aggregate (MV) | Daily revenue KPIs by tier x product -- AOV, net revenue, refund rate |
| `agg_user_activity` | Aggregate (MV) | Daily engagement KPIs by tier x product -- DAU, error rate, latency P95 |

> **Analyst note:** Use `dim_user WHERE __END_AT IS NULL` to get current user state.
> The SCD2 history lets you answer questions like: what tier was this customer in when they churned?

## 1. Setup -- Connect to Databricks (Serverless)

In [1]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.serverless(True).profile("DEFAULT").getOrCreate()

CATALOG = "sv_ai_builder_workspace_catalog"
GOLD    = "saas_gold_dev"
SANDBOX = "saas_analysis_sandbox"

print(f"Connected to Databricks  (Spark {spark.version})")
print(f"  Gold schema  : {CATALOG}.{GOLD}")
print(f"  Sandbox      : {CATALOG}.{SANDBOX}")

# Helper: print a section header and run a SQL query
def sql_show(query, title="", n=5):
    if title:
        print(f"\n{'=' * 70}")
        print(f"  {title}")
        print(f"{'=' * 70}")
    spark.sql(query).show(n, truncate=50)

Connected to Databricks  (Spark 4.1.0)
  Gold schema  : sv_ai_builder_workspace_catalog.saas_gold_dev
  Sandbox      : sv_ai_builder_workspace_catalog.saas_analysis_sandbox


## 2. Create Analysis Sandbox Schema

In [2]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SANDBOX}")
spark.sql(f"USE CATALOG {CATALOG}")

print(f"Schema ready: {CATALOG}.{SANDBOX}")
print(f"\nGold tables:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD}").show(truncate=False)

Schema ready: sv_ai_builder_workspace_catalog.saas_analysis_sandbox

Gold tables:
+-------------+-----------------+-----------+
|database     |tableName        |isTemporary|
+-------------+-----------------+-----------+
|saas_gold_dev|agg_daily_revenue|false      |
|saas_gold_dev|agg_user_activity|false      |
|saas_gold_dev|dim_user         |false      |
|saas_gold_dev|fact_event       |false      |
|saas_gold_dev|fact_order       |false      |
+-------------+-----------------+-----------+



---

## 3. `dim_user` -- Current User Dimension (SCD Type 2)

**What it contains:** One row per active version of each user. Because this is SCD Type 2,
subscription tier changes, product switches, and deactivations each create a new row --
the old row gets its `__END_AT` timestamp filled in.

**Key SaaS questions it answers:**
- What is our current active user base by subscription tier?
- Which companies are on Enterprise plans?
- How has a user's plan changed over time?
- What % of users have MFA enabled (security posture)?

**Analyst tip:** `WHERE __END_AT IS NULL` = current state. Use `__START_AT` / `__END_AT`
for historical point-in-time analysis.

In [3]:
# Row counts: total vs current vs historical
spark.sql(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT_IF(__END_AT IS NULL)      AS current_rows,
        COUNT_IF(__END_AT IS NOT NULL)  AS historical_rows
    FROM {CATALOG}.{GOLD}.dim_user
""").show()

# Top 5 current users -- business-relevant columns
sql_show(f"""
    SELECT
        user_id, full_name, company, role,
        subscription_tier, product_type,
        country, city,
        is_active, is_enterprise, mfa_enabled,
        account_age_days, __START_AT
    FROM {CATALOG}.{GOLD}.dim_user
    WHERE __END_AT IS NULL
    ORDER BY user_id
    LIMIT 5
""", title="dim_user -- Top 5 Current User Records (SCD2, __END_AT IS NULL)")

+----------+------------+---------------+
|total_rows|current_rows|historical_rows|
+----------+------------+---------------+
|     10000|       10000|              0|
+----------+------------+---------------+


  dim_user -- Top 5 Current User Records (SCD2, __END_AT IS NULL)
+----------+---------------+-----------------------------+-------+-----------------+--------------+-------+-------------------+---------+-------------+-----------+----------------+-----------------------+
|   user_id|      full_name|                      company|   role|subscription_tier|  product_type|country|               city|is_active|is_enterprise|mfa_enabled|account_age_days|             __START_AT|
+----------+---------------+-----------------------------+-------+-----------------+--------------+-------+-------------------+---------+-------------+-----------+----------------+-----------------------+
|USR-000000|     Brian Yang|Miller, Henderson and Johnson|analyst|          Starter|  BI Dashboard|     JP|

---

## 4. `fact_order` -- Order Fact Table (SCD Type 1)

**What it contains:** One row per unique `order_id`, always showing the latest status.
Status transitions (e.g., `pending -> completed -> refunded`) update the existing row
in-place -- no duplicates.

**Key SaaS questions it answers:**
- What is our total bookings volume by product type?
- What is the mix of subscription tiers across orders?
- What % of orders are high-value (>$500)?
- Which payment methods are most popular?
- How many orders have been refunded (early churn signal)?

**Analyst tip:** Join to `dim_user` on `user_id WHERE __END_AT IS NULL` for the user's
current tier, or join on `order_date BETWEEN __START_AT AND __END_AT` for the tier
they held at time of purchase.

In [4]:
spark.sql(f"SELECT COUNT(*) AS total_orders FROM {CATALOG}.{GOLD}.fact_order").show()

print("Order status distribution:")
spark.sql(f"""
    SELECT status, COUNT(*) AS order_count
    FROM {CATALOG}.{GOLD}.fact_order
    GROUP BY status
    ORDER BY order_count DESC
""").show()

sql_show(f"""
    SELECT
        order_id, user_id, product_type, subscription_tier,
        unit_price, quantity, total_value, order_category,
        is_high_value, status, payment_method,
        currency, order_date, order_quarter
    FROM {CATALOG}.{GOLD}.fact_order
    ORDER BY order_id
    LIMIT 5
""", title="fact_order -- Top 5 Orders (SCD1, latest state per order_id)")

+------------+
|total_orders|
+------------+
|       30000|
+------------+

Order status distribution:
+---------+-----------+
|   status|order_count|
+---------+-----------+
|completed|      24578|
|  pending|       2930|
|cancelled|       1521|
| refunded|        971|
+---------+-----------+


  fact_order -- Top 5 Orders (SCD1, latest state per order_id)
+-----------+----------+------------------+-----------------+----------+--------+-----------+--------------+-------------+---------+--------------+--------+----------+-------------+
|   order_id|   user_id|      product_type|subscription_tier|unit_price|quantity|total_value|order_category|is_high_value|   status|payment_method|currency|order_date|order_quarter|
+-----------+----------+------------------+-----------------+----------+--------+-----------+--------------+-------------+---------+--------------+--------+----------+-------------+
|ORD-0000000|USR-007323|      ML Workbench|             Free|     32.10|       1|      32.10| 

---

## 5. `fact_event` -- Event Fact Table (SCD Type 1 / Deduplicated)

**What it contains:** One row per unique `event_id` after deduplication. Covers all web app
activity: page views, feature usage, API calls, errors, and more. This is the foundation
for product analytics.

**Key SaaS questions it answers:**
- Which features are most (and least) used?
- What is our error rate by product/tier?
- Which user actions have the highest latency (product performance)?
- What is the breakdown of events by category (navigation vs feature use vs API)?
- Which sessions had a high proportion of errors?

**Analyst tip:** `event_category` classifies events into `navigation`, `api_call`,
`feature_use`, and `system_error` -- use it for quick funnel segmentation without
touching raw `event_type` strings.

In [5]:
spark.sql(f"SELECT COUNT(*) AS total_events FROM {CATALOG}.{GOLD}.fact_event").show()

print("Event category distribution:")
spark.sql(f"""
    SELECT event_category, COUNT(*) AS event_count
    FROM {CATALOG}.{GOLD}.fact_event
    GROUP BY event_category
    ORDER BY event_count DESC
""").show()

sql_show(f"""
    SELECT
        event_id, user_id, session_id,
        event_type, event_category,
        product_type, subscription_tier,
        feature_name, latency_ms, is_high_latency,
        http_status_code, is_error_event,
        event_date, event_hour, country, device_type
    FROM {CATALOG}.{GOLD}.fact_event
    ORDER BY event_timestamp
    LIMIT 5
""", title="fact_event -- Top 5 Events (SCD1, deduplicated by event_id)")

+------------+
|total_events|
+------------+
|       50000|
+------------+

Event category distribution:
+--------------+-----------+
|event_category|event_count|
+--------------+-----------+
|         other|      22953|
|    navigation|      19618|
|      api_call|       3785|
|   feature_use|       2224|
|  system_error|       1420|
+--------------+-----------+


  fact_event -- Top 5 Events (SCD1, deduplicated by event_id)
+------------+----------+-----------------+-----------+--------------+------------------+-----------------+------------+----------+---------------+----------------+--------------+----------+----------+-------+-----------+
|    event_id|   user_id|       session_id| event_type|event_category|      product_type|subscription_tier|feature_name|latency_ms|is_high_latency|http_status_code|is_error_event|event_date|event_hour|country|device_type|
+------------+----------+-----------------+-----------+--------------+------------------+-----------------+------------+------

---

## 6. `agg_daily_revenue` -- Daily Revenue Aggregate (Materialized View)

**What it contains:** Pre-aggregated daily revenue metrics, sliced by `subscription_tier`
and `product_type`. Built directly from `fact_order` on every pipeline run.

**Key SaaS questions it answers (without writing SQL joins):**
- What is our daily MRR trend by tier?
- Which product x tier combination drives the most revenue?
- Is our average order value (AOV) trending up or down?
- What is our daily refund rate -- are there spikes?
- How many high-value (>$500) deals close per day?

**Metrics key:**
`total_revenue` = gross | `net_revenue` = completed only | `refund_rate` = refunded/total | `unique_customers` = DAC

In [6]:
spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(order_date) AS earliest_date,
        MAX(order_date) AS latest_date
    FROM {CATALOG}.{GOLD}.agg_daily_revenue
""").show()

sql_show(f"""
    SELECT
        order_date, subscription_tier, product_type,
        order_count, unique_customers,
        total_revenue, net_revenue, avg_order_value,
        high_value_count, completed_orders, refunded_orders, refund_rate
    FROM {CATALOG}.{GOLD}.agg_daily_revenue
    ORDER BY order_date DESC
    LIMIT 5
""", title="agg_daily_revenue -- Top 5 Most Recent Days (Materialized View)")

+----------+-------------+-----------+
|total_rows|earliest_date|latest_date|
+----------+-------------+-----------+
|      3554|   2025-11-27| 2026-02-25|
+----------+-------------+-----------+


  agg_daily_revenue -- Top 5 Most Recent Days (Materialized View)
+----------+-----------------+------------------+-----------+----------------+-------------+-----------+---------------+----------------+----------------+---------------+-----------+
|order_date|subscription_tier|      product_type|order_count|unique_customers|total_revenue|net_revenue|avg_order_value|high_value_count|completed_orders|refunded_orders|refund_rate|
+----------+-----------------+------------------+-----------+----------------+-------------+-----------+---------------+----------------+----------------+---------------+-----------+
|2026-02-25|     Professional|    Security Suite|          4|               4|       517.90|     517.90|         129.48|               0|               4|              0|        0.0|
|2026

---

## 7. `agg_user_activity` -- Daily Engagement Aggregate (Materialized View)

**What it contains:** Pre-aggregated daily engagement metrics, sliced by `subscription_tier`
and `product_type`. Built from `fact_event` on every pipeline run.

**Key SaaS questions it answers:**
- What is our DAU trend? (are we growing engaged users?)
- What is the error rate by tier -- are Enterprise customers experiencing more errors?
- Is our P95 latency within SLO (typically <2s for web apps)?
- Are feature_use events growing faster than navigation events? (product depth metric)
- How many API calls per day -- which tiers use integrations most?

**Metrics key:**
`dau` = Daily Active Users | `error_rate` = errors/total events | `p95_latency_ms` = SLO metric | `high_latency_pct` = % events >1s

In [7]:
spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(event_date) AS earliest_date,
        MAX(event_date) AS latest_date
    FROM {CATALOG}.{GOLD}.agg_user_activity
""").show()

sql_show(f"""
    SELECT
        event_date, subscription_tier, product_type,
        dau, total_events, total_sessions,
        avg_latency_ms, p95_latency_ms, high_latency_pct,
        error_count, error_rate,
        feature_use_count, api_call_count
    FROM {CATALOG}.{GOLD}.agg_user_activity
    ORDER BY event_date DESC
    LIMIT 5
""", title="agg_user_activity -- Top 5 Most Recent Days (Materialized View)")

+----------+-------------+-----------+
|total_rows|earliest_date|latest_date|
+----------+-------------+-----------+
|        40|   2026-02-25| 2026-02-25|
+----------+-------------+-----------+


  agg_user_activity -- Top 5 Most Recent Days (Materialized View)
+----------+-----------------+------------------+---+------------+--------------+--------------+--------------+----------------+-----------+----------+-----------------+--------------+
|event_date|subscription_tier|      product_type|dau|total_events|total_sessions|avg_latency_ms|p95_latency_ms|high_latency_pct|error_count|error_rate|feature_use_count|api_call_count|
+----------+-----------------+------------------+---+------------+--------------+--------------+--------------+----------------+-----------+----------+-----------------+--------------+
|2026-02-25|          Starter|      BI Dashboard|212|        1045|           324|        203.66|           454|          0.0278|         32|    0.0306|               47|            7

---

## 8. Schema Summary -- What's Available in the Sandbox

Save any intermediate analysis tables here for downstream use with
`CREATE TABLE ... AS SELECT` or `df.write.saveAsTable(...)`.

In [8]:
spark.sql(f"""
    SELECT
        table_name,
        table_type,
        data_source_format,
        table_owner,
        created,
        last_altered
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = '{GOLD}'
      AND table_name NOT LIKE '__materialization%'
    ORDER BY table_name
""").show(truncate=False)

print(f"Analysis sandbox : {CATALOG}.{SANDBOX}")
print(f"Save results with:")
print(f"  spark.sql(\"CREATE OR REPLACE TABLE {CATALOG}.{SANDBOX}.<name> AS SELECT ...\")") 

+-----------------+-----------------+--------------------------+------------------------------+-----------------------+-----------------------+
|table_name       |table_type       |data_source_format        |table_owner                   |created                |last_altered           |
+-----------------+-----------------+--------------------------+------------------------------+-----------------------+-----------------------+
|agg_daily_revenue|MATERIALIZED_VIEW|UNKNOWN_DATA_SOURCE_FORMAT|saroj.venkatesh@databricks.com|2026-02-26 05:18:31.675|2026-02-26 05:19:12.012|
|agg_user_activity|MATERIALIZED_VIEW|UNKNOWN_DATA_SOURCE_FORMAT|saroj.venkatesh@databricks.com|2026-02-26 05:18:31.675|2026-02-26 05:19:11.995|
|dim_user         |STREAMING_TABLE  |UNKNOWN_DATA_SOURCE_FORMAT|saroj.venkatesh@databricks.com|2026-02-26 05:18:31.673|2026-02-26 05:18:35.187|
|fact_event       |STREAMING_TABLE  |UNKNOWN_DATA_SOURCE_FORMAT|saroj.venkatesh@databricks.com|2026-02-26 05:18:31.676|2026-02-26 05:18:

---

## 9. Monthly Sales Trend -- Last 3 Months

Revenue and paying customers by subscription tier for **Dec 2025, Jan 2026, Feb 2026**.
Free tier users who placed orders are included; Feb 2026 is a partial month (through the 25th).

> **Tiers ranked by revenue:** Enterprise >> Business >> Professional >> Starter > Free

In [10]:
# Last 3 complete months: start from 2 months ago (Dec 1) to capture Dec / Jan / Feb
sql_show(f"""
    SELECT
        DATE_FORMAT(order_date, 'yyyy-MM')                                          AS month,
        subscription_tier,
        COUNT(DISTINCT user_id)                                                     AS paying_customers,
        COUNT(order_id)                                                             AS orders,
        CAST(SUM(total_value)                                        AS DECIMAL(18,2)) AS gross_revenue,
        CAST(SUM(CASE WHEN status = 'completed' THEN total_value ELSE 0 END) AS DECIMAL(18,2)) AS net_revenue,
        SUM(CASE WHEN status = 'refunded' THEN 1 ELSE 0 END)                        AS refunds
    FROM {CATALOG}.{GOLD}.fact_order
    WHERE order_date >= DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -2))
      AND status != 'cancelled'
    GROUP BY DATE_FORMAT(order_date, 'yyyy-MM'), subscription_tier
    ORDER BY month DESC, gross_revenue DESC
""", title="Monthly Sales Trend -- Paying Customers & Revenue by Tier (last 3 months)", n=20)


  Monthly Sales Trend -- Paying Customers & Revenue by Tier (last 3 months)
+-------+-----------------+----------------+------+-------------+-----------+-------+
|  month|subscription_tier|paying_customers|orders|gross_revenue|net_revenue|refunds|
+-------+-----------------+----------------+------+-------------+-----------+-------+
|2026-02|       Enterprise|             709|  1976|   2672296.13| 2255937.75|     73|
|2026-02|         Business|            1148|  2216|   1218859.16| 1041376.59|     76|
|2026-02|     Professional|            1250|  1852|    308352.83|  261277.53|     61|
|2026-02|          Starter|             910|  1127|     63576.24|   54491.81|     39|
|2026-02|             Free|             528|   575|     38176.76|   33193.05|     15|
|2026-01|       Enterprise|             746|  2539|   3400572.81| 2922113.67|     86|
|2026-01|         Business|            1278|  2868|   1573333.81| 1353079.51|    102|
|2026-01|     Professional|            1409|  2313|    386493.1

---

## 10. Monthly Renewal Rate

**Definition:** Renewal rate = customers who purchased in both month N-1 and month N, divided by all customers active in month N-1.
A customer "renews" if they placed a non-cancelled order in two consecutive months.

> Note: Dec 2025 renewal rate is inflated because Nov 2025 had only 5 days of data (~1,016 customers at launch vs. 5,000+ steady state). Jan and Feb 2026 reflect steady-state rates.

In [11]:
# -- Overall renewal rate trend --
sql_show(f"""
    WITH monthly_buyers AS (
        SELECT DISTINCT DATE_TRUNC('MONTH', order_date) AS month, user_id
        FROM {CATALOG}.{GOLD}.fact_order WHERE status != 'cancelled'
    ),
    base AS (
        SELECT curr.month,
               COUNT(DISTINCT curr.user_id) AS active,
               COUNT(DISTINCT CASE WHEN prev.user_id IS NOT NULL THEN curr.user_id END) AS renewed
        FROM monthly_buyers curr
        LEFT JOIN monthly_buyers prev
               ON curr.user_id = prev.user_id
              AND curr.month = ADD_MONTHS(prev.month, 1)
        GROUP BY curr.month
    ),
    lagged AS (
        SELECT month, active, renewed, LAG(active) OVER (ORDER BY month) AS prev_active FROM base
    )
    SELECT DATE_FORMAT(month, 'yyyy-MM')                                              AS month,
           prev_active                                                                 AS customers_up_for_renewal,
           renewed                                                                     AS renewed_customers,
           active - renewed                                                            AS new_customers,
           active                                                                      AS total_active,
           CONCAT(CAST(ROUND(100.0 * renewed / NULLIF(prev_active, 0), 1) AS STRING), '%') AS renewal_rate
    FROM lagged ORDER BY month DESC LIMIT 5
""", title="Overall Monthly Renewal Rate")

# -- Renewal rate by subscription tier (last 2 full months) --
sql_show(f"""
    WITH monthly_buyers AS (
        SELECT DISTINCT DATE_TRUNC('MONTH', order_date) AS month, user_id, subscription_tier
        FROM {CATALOG}.{GOLD}.fact_order WHERE status != 'cancelled'
    )
    SELECT DATE_FORMAT(curr.month, 'yyyy-MM') AS month,
           curr.subscription_tier,
           COUNT(DISTINCT CASE WHEN prev.user_id IS NOT NULL THEN curr.user_id END) AS renewed,
           COUNT(DISTINCT curr.user_id)                                              AS active,
           CONCAT(CAST(ROUND(
               100.0 * COUNT(DISTINCT CASE WHEN prev.user_id IS NOT NULL THEN curr.user_id END)
               / NULLIF(COUNT(DISTINCT curr.user_id), 0), 1) AS STRING), '%')        AS returning_pct
    FROM monthly_buyers curr
    LEFT JOIN monthly_buyers prev
           ON curr.user_id = prev.user_id
          AND curr.month = ADD_MONTHS(prev.month, 1)
    WHERE curr.month >= DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -2))
    GROUP BY curr.month, curr.subscription_tier
    ORDER BY month DESC, active DESC
""", title="Renewal Rate by Subscription Tier (last 3 months)")


  Overall Monthly Renewal Rate
+-------+------------------------+-----------------+-------------+------------+------------+
|  month|customers_up_for_renewal|renewed_customers|new_customers|total_active|renewal_rate|
+-------+------------------------+-----------------+-------------+------------+------------+
|2026-02|                    5184|             2974|         1571|        4545|       57.4%|
|2026-01|                    5160|             3344|         1840|        5184|       64.8%|
|2025-12|                    1016|              698|         4462|        5160|       68.7%|
|2025-11|                    NULL|                0|         1016|        1016|        NULL|
+-------+------------------------+-----------------+-------------+------------+------------+


  Renewal Rate by Subscription Tier (last 3 months)
+-------+-----------------+-------+------+-------------+
|  month|subscription_tier|renewed|active|returning_pct|
+-------+-----------------+-------+------+-------------+

---

## 11. Churn Risk Analysis

Three leading indicators are analysed across the user base:

| Signal | Why it matters |
|--------|---------------|
| **Recency** | Days since last order -- the single strongest predictor of churn |
| **Product friction** | High error rates and latency create frustration before a customer decides to leave |
| **At-risk profile** | Combined view of inactive users: who they are, how broken their experience is |

> **Refund nuance:** Users with 1+ refunds actually order *more* (avg 6.6 vs 5.2 orders for Business) and lapsed *sooner* after a refund. Refunds indicate heavy users pushing limits -- not a direct churn signal in this dataset, but worth monitoring alongside inactivity.

In [12]:
# Signal 1: Churn risk by order recency -- how many users in each tier have gone quiet?
sql_show(f"""
    WITH last_order AS (
        SELECT user_id, subscription_tier,
               MAX(order_date)  AS last_order_date,
               COUNT(order_id)  AS lifetime_orders
        FROM {CATALOG}.{GOLD}.fact_order
        GROUP BY user_id, subscription_tier
    )
    SELECT subscription_tier,
           CASE
               WHEN DATEDIFF(CURRENT_DATE(), last_order_date) <= 30 THEN '1. Active     (0-30d)'
               WHEN DATEDIFF(CURRENT_DATE(), last_order_date) <= 60 THEN '2. At Risk    (31-60d)'
               WHEN DATEDIFF(CURRENT_DATE(), last_order_date) <= 90 THEN '3. High Risk  (61-90d)'
               ELSE                                                       '4. Churned    (90d+)'
           END                                                          AS churn_risk,
           COUNT(*)                                                      AS users,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY subscription_tier), 1) AS pct_of_tier
    FROM last_order
    GROUP BY subscription_tier, churn_risk
    ORDER BY subscription_tier, churn_risk
""", title="Signal 1 -- Churn Risk Segments by Subscription Tier (order recency)", n=25)


  Signal 1 -- Churn Risk Segments by Subscription Tier (order recency)
+-----------------+----------------------+-----+-----------+
|subscription_tier|            churn_risk|users|pct_of_tier|
+-----------------+----------------------+-----+-----------+
|         Business| 1. Active     (0-30d)| 1262|       83.7|
|         Business|2. At Risk    (31-60d)|  205|       13.6|
|         Business|3. High Risk  (61-90d)|   40|        2.7|
|       Enterprise| 1. Active     (0-30d)|  746|       96.5|
|       Enterprise|2. At Risk    (31-60d)|   27|        3.5|
|             Free| 1. Active     (0-30d)|  647|       40.8|
|             Free|2. At Risk    (31-60d)|  528|       33.3|
|             Free|3. High Risk  (61-90d)|  401|       25.3|
|             Free|  4. Churned    (90d+)|    8|        0.5|
|     Professional| 1. Active     (0-30d)| 1411|       67.2|
|     Professional|2. At Risk    (31-60d)|  511|       24.3|
|     Professional|3. High Risk  (61-90d)|  177|        8.4|
|     Profess

In [13]:
# Signal 2: Product friction -- error rate and latency by tier (experience problems precede churn)
sql_show(f"""
    SELECT u.subscription_tier,
           COUNT(DISTINCT e.user_id)                                                               AS users_with_events,
           ROUND(100.0 * SUM(CASE WHEN e.is_error_event  THEN 1 ELSE 0 END) / COUNT(e.event_id), 2) AS error_rate_pct,
           ROUND(100.0 * SUM(CASE WHEN e.is_high_latency THEN 1 ELSE 0 END) / COUNT(e.event_id), 2) AS high_latency_pct,
           ROUND(AVG(e.latency_ms), 0)                                                             AS avg_latency_ms,
           ROUND(PERCENTILE_APPROX(e.latency_ms, 0.95), 0)                                         AS p95_latency_ms
    FROM {CATALOG}.{GOLD}.fact_event e
    JOIN {CATALOG}.{GOLD}.dim_user u ON e.user_id = u.user_id AND u.__END_AT IS NULL
    GROUP BY u.subscription_tier
    ORDER BY error_rate_pct DESC
""", title="Signal 2 -- Product Friction by Tier (error rate + latency; Free tier leads on both)")


  Signal 2 -- Product Friction by Tier (error rate + latency; Free tier leads on both)
+-----------------+-----------------+--------------+----------------+--------------+--------------+
|subscription_tier|users_with_events|error_rate_pct|high_latency_pct|avg_latency_ms|p95_latency_ms|
+-----------------+-----------------+--------------+----------------+--------------+--------------+
|             Free|             1463|          3.70|            2.72|         202.0|           485|
|     Professional|             1207|          3.53|            2.68|         204.0|           516|
|         Business|              858|          3.39|            2.55|         198.0|           461|
|          Starter|             1454|          3.36|            2.78|         202.0|           503|
|       Enterprise|              448|          3.32|            2.59|         203.0|           478|
+-----------------+-----------------+--------------+----------------+--------------+--------------+



In [14]:
# Signal 3: At-risk user profile -- who has gone quiet AND is experiencing friction?
# Inactive 31-90 days, grouped by tier to show scale of exposure and experience quality
sql_show(f"""
    WITH order_recency AS (
        SELECT user_id, subscription_tier,
               DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS days_inactive,
               COUNT(order_id)                           AS lifetime_orders
        FROM {CATALOG}.{GOLD}.fact_order
        WHERE status != 'cancelled'
        GROUP BY user_id, subscription_tier
    ),
    user_events AS (
        SELECT user_id,
               COUNT(*)                                                              AS total_events,
               ROUND(100.0 * SUM(CASE WHEN is_error_event  THEN 1 ELSE 0 END) / COUNT(*), 2) AS error_rate_pct,
               ROUND(100.0 * SUM(CASE WHEN is_high_latency THEN 1 ELSE 0 END) / COUNT(*), 2) AS latency_pct
        FROM {CATALOG}.{GOLD}.fact_event
        GROUP BY user_id
    )
    SELECT o.subscription_tier,
           COUNT(*)                                  AS at_risk_users,
           ROUND(AVG(o.days_inactive), 0)            AS avg_days_inactive,
           ROUND(AVG(o.lifetime_orders), 1)          AS avg_lifetime_orders,
           ROUND(AVG(COALESCE(e.error_rate_pct, 0)), 2) AS avg_error_rate_pct,
           ROUND(AVG(COALESCE(e.latency_pct,    0)), 2) AS avg_high_latency_pct,
           ROUND(AVG(COALESCE(e.total_events,   0)), 0) AS avg_events
    FROM order_recency o
    LEFT JOIN user_events e ON o.user_id = e.user_id
    WHERE o.days_inactive BETWEEN 31 AND 90
    GROUP BY o.subscription_tier
    ORDER BY at_risk_users DESC
""", title="Signal 3 -- At-Risk User Profile (31-90 days inactive): tier, friction & engagement depth")


  Signal 3 -- At-Risk User Profile (31-90 days inactive): tier, friction & engagement depth
+-----------------+-------------+-----------------+-------------------+------------------+--------------------+----------+
|subscription_tier|at_risk_users|avg_days_inactive|avg_lifetime_orders|avg_error_rate_pct|avg_high_latency_pct|avg_events|
+-----------------+-------------+-----------------+-------------------+------------------+--------------------+----------+
|          Starter|         1007|             55.0|                1.6|              1.69|                1.41|       3.0|
|             Free|          896|             58.0|                1.3|              1.97|                1.67|       1.0|
|     Professional|          707|             50.0|                2.4|              2.04|                1.67|       5.0|
|         Business|          257|             46.0|                3.8|              1.81|                1.27|      10.0|
|       Enterprise|           37|             